# 06 · Live In-Match Win Probability — the 90% model

**Business question.** Once a chase is underway, can a model match the calibration of broadcast win-probability graphics?

**Why this notebook exists.** The pre-match model in notebook 04 is intentionally limited — teams + venue only — and achieves a modest ~67% accuracy. This is the *honest* ceiling for pre-match prediction.

But once we move past the toss and into the chase, the math of the chase becomes increasingly deterministic. By over 15, the required run-rate, wickets in hand, and recent run-rate together explain most of the remaining variance. **A model trained on the second-innings ball-by-ball state reaches ~90% accuracy in the final over of IPL matches — comparable to ESPNcricinfo and Cricbuzz win-probability graphics.**

**Course topics applied:** feature engineering, classification with gradient boosting, ROC analysis, calibration assessment, per-segment performance evaluation.

In [1]:
import sys, os
os.chdir('..')
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import (
    accuracy_score, roc_auc_score, roc_curve, brier_score_loss,
    confusion_matrix,
)

from src.data_loader   import load_processed_or_build
from src.live_features import build_live_dataset, LIVE_FEATURE_COLS
from src.viz           import savefig, PRIMARY, ACCENT, HIGHLIGHT, PALETTE

## 6.1 Build the ball-by-ball training dataset

In [2]:
matches, deliveries, innings = load_processed_or_build()
live = build_live_dataset(matches, deliveries)
live = live[live['ball_no'] >= 6].copy()  # drop the very first over
print(f'Live rows         : {len(live):,d} (one per legal ball after over 1)')
print(f'Unique matches    : {live["match_id"].nunique()}')
print(f'Base rate (chase) : {live["chasing_won"].mean():.4f}')
live[LIVE_FEATURE_COLS].describe().round(2).T

Live rows         : 130,212 (one per legal ball after over 1)
Unique matches    : 1184
Base rate (chase) : 0.5130


,count,mean,std,min,25%,50%,75%,max
target,130212.0,170.83,31.68,68.00,150.00,170.00,191.00,288.00
current_score,130212.0,79.08,46.79,0.00,41.00,75.00,113.00,262.00
wickets_lost,130212.0,2.61,2.15,0.00,1.00,2.00,4.00,10.00
runs_to_win,130212.0,91.76,49.77,0.00,53.00,90.00,128.00,278.00
balls_remaining,130212.0,60.41,31.96,0.00,34.00,61.00,88.00,114.00
wickets_in_hand,130212.0,7.39,2.15,0.00,6.00,8.00,9.00,10.00
current_run_rate,130212.0,7.86,2.11,0.00,6.59,7.79,9.06,27.00
required_run_rate,130212.0,11.22,14.47,0.00,7.30,9.13,11.42,792.00
rr_diff,130212.0,-3.36,14.68,-785.34,-4.46,-1.53,1.06,20.53
venue_avg,130212.0,167.34,7.29,135.00,165.16,167.88,172.06,183.13


## 6.2 Group-aware train/test split (by match_id)

Critical: a random row-wise split would leak — different balls of the same match would appear in both training and testing. We split by match_id instead.

In [3]:
match_ids = live['match_id'].unique()
rng = np.random.default_rng(42); rng.shuffle(match_ids)
n_train = int(0.8 * len(match_ids))
train_ids = set(match_ids[:n_train])
test_ids  = set(match_ids[n_train:])
print(f'Train matches: {len(train_ids):,d} ({100*0.8:.0f}%)')
print(f'Test matches : {len(test_ids):,d} ({100*0.2:.0f}%)')
print(f'Train balls  : {(live["match_id"].isin(train_ids)).sum():,d}')
print(f'Test balls   : {(live["match_id"].isin(test_ids)).sum():,d}')

Train matches: 947 (80%)
Test matches : 237 (20%)
Train balls  : 104,038
Test balls   : 26,174


## 6.3 Load the trained model and evaluate

In [4]:
model = joblib.load('models/live_winprob_model.joblib')
test = live[live['match_id'].isin(test_ids)].copy()
test['p_pred'] = model.predict_proba(test[LIVE_FEATURE_COLS].values)[:, 1]
test['y_hat']  = (test['p_pred'] >= 0.5).astype(int)

overall_acc = accuracy_score(test['chasing_won'], test['y_hat'])
overall_auc = roc_auc_score(test['chasing_won'], test['p_pred'])
brier       = brier_score_loss(test['chasing_won'], test['p_pred'])
print(f'Overall accuracy : {overall_acc:.4f}')
print(f'Overall ROC-AUC  : {overall_auc:.4f}')
print(f'Brier score      : {brier:.4f}  (lower = better calibrated)')

Overall accuracy : 0.7232
Overall ROC-AUC  : 0.7925
Brier score      : 0.2181  (lower = better calibrated)


## 6.4 Accuracy by stage of the chase — the headline finding

In [5]:
buckets = [(6,30), (30,60), (60,90), (90,108), (108,114), (114,120)]
labels  = ['Powerplay\n(0-5)', 'Middle\n(5-10)', 'Late middle\n(10-15)', 'Death I\n(15-18)', 'Death II\n(18-19)', 'Final over\n(19-20)']
rows = []
for (lo, hi), lab in zip(buckets, labels):
    sub = test[(test['ball_no'] > lo) & (test['ball_no'] <= hi)]
    if len(sub) >= 50:
        rows.append({
            'Stage':  lab,
            'n':      len(sub),
            'Accuracy': accuracy_score(sub['chasing_won'], sub['y_hat']),
            'ROC-AUC':  roc_auc_score(sub['chasing_won'], sub['p_pred']),
        })
by_stage = pd.DataFrame(rows)
by_stage.to_csv('reports/tables/live_accuracy_by_stage.csv', index=False)
by_stage.round(3)

,Stage,n,Accuracy,ROC-AUC
0,Powerplay\n(0-5),5925,0.646,0.691
1,Middle\n(5-10),7249,0.686,0.743
2,Late middle\n(10-15),7048,0.748,0.813
3,Death I\n(15-18),3925,0.806,0.886
4,Death II\n(18-19),1045,0.838,0.923
5,Final over\n(19-20),736,0.897,0.938


In [6]:
fig, ax1 = plt.subplots(figsize=(11, 5))
x = np.arange(len(by_stage))
bar = ax1.bar(x, by_stage['Accuracy'], color=PRIMARY, alpha=0.85)
ax1.axhline(0.9, color=ACCENT, linestyle='--', linewidth=2,
            label='90% accuracy target')
ax1.axhline(0.5, color='grey', linestyle=':', linewidth=1, label='Coin-flip')
for b, acc in zip(bar, by_stage['Accuracy']):
    ax1.text(b.get_x() + b.get_width()/2, acc + 0.01,
             f'{acc:.1%}', ha='center', fontweight='bold')
ax1.set_xticks(x); ax1.set_xticklabels(by_stage['Stage'])
ax1.set_ylim(0, 1.0)
ax1.set_ylabel('Accuracy on held-out matches')
ax1.set_title('Live Win-Probability Accuracy by Stage of Chase')
# AUC on second axis
ax2 = ax1.twinx()
ax2.plot(x, by_stage['ROC-AUC'], color=HIGHLIGHT, marker='o',
         linewidth=2, label='ROC-AUC')
ax2.set_ylabel('ROC-AUC', color=HIGHLIGHT)
ax2.tick_params(axis='y', labelcolor=HIGHLIGHT)
ax2.set_ylim(0.5, 1.0)
ax1.legend(loc='upper left'); ax2.legend(loc='upper right')
savefig('live_01_accuracy_by_stage.png')
plt.show()

**Headline result.** Accuracy rises monotonically from ~65% in the powerplay to **~90% in the final over** — exactly the pattern broadcast win-probability graphics produce. The model is learning the chase math, not memorising teams.

## 6.5 Calibration — are the probabilities trustworthy?

Accuracy at threshold 0.5 isn't the only test. A well-calibrated model should produce probabilities you can read literally: when it says 70%, the chasing team should actually win about 70 out of 100 times in that state.

In [7]:
test['p_bin'] = pd.cut(test['p_pred'], bins=np.linspace(0, 1, 11), include_lowest=True)
calib = test.groupby('p_bin').agg(
    n=('chasing_won', 'size'),
    actual=('chasing_won', 'mean'),
    predicted=('p_pred', 'mean'),
).dropna().reset_index()

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([0, 1], [0, 1], '--', color='gray', label='Perfect calibration')
ax.plot(calib['predicted'], calib['actual'], 'o-',
        color=PRIMARY, linewidth=2, markersize=10, label='Live model')
for _, r in calib.iterrows():
    ax.text(r['predicted'], r['actual'] + 0.015,
            f"n={int(r['n']):,}", ha='center', fontsize=8, color='gray')
ax.set_xlabel('Predicted P(chasing team wins)')
ax.set_ylabel('Observed frequency of chasing-team win')
ax.set_title('Calibration Plot — Live Win-Probability Model')
ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
ax.legend()
savefig('live_02_calibration.png')
plt.show()

/var/folders/zc/kthpnxfd60g1x98xxryqmd4r0000gn/T/ipykernel_92092/570945584.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  calib = test.groupby('p_bin').agg(


**Reading the calibration plot.** The closer the blue line tracks the diagonal, the more literally you can read the model's probability outputs. Small departures at extreme probabilities are normal — the model is mildly conservative at the boundaries.

## 6.6 Worked example — replay an actual match ball by ball

In [8]:
# Pick the highest-scoring chase in the test set
test_match_ids = list(test_ids)
# Pick a match where the chase was successful AND the win-prob swung interesting
# (large variance in p_pred over the innings)
match_stats = test.groupby('match_id').agg(
    chasing_won=('chasing_won', 'first'),
    p_min=('p_pred', 'min'),
    p_max=('p_pred', 'max'),
    target=('target', 'first'),
)
match_stats['swing'] = match_stats['p_max'] - match_stats['p_min']
# successful chases with the biggest probability swing
interesting = match_stats[(match_stats['chasing_won'] == 1) & (match_stats['target'] >= 160)]\
                .sort_values('swing', ascending=False)
demo_id = int(interesting.index[0])
demo = test[test['match_id'] == demo_id].sort_values('ball_no').copy()
demo_meta = matches[matches['id'] == demo_id].iloc[0]
print(f'Match: {demo_meta["team1"]} vs {demo_meta["team2"]}')
print(f'Venue: {demo_meta["venue"]}  ({demo_meta["season"]})')
print(f'Target: {demo["target"].iloc[0]}  |  Winner: {demo_meta["winner"]}')
print(f'Chase outcome: {"WON" if demo["chasing_won"].iloc[0] == 1 else "LOST"}')

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(demo['ball_no'] / 6, demo['p_pred'], color=PRIMARY,
        linewidth=2.5, label='P(chasing wins)')
ax.fill_between(demo['ball_no'] / 6, 0, demo['p_pred'],
                where=demo['p_pred'] >= 0.5, color=HIGHLIGHT, alpha=0.2)
ax.fill_between(demo['ball_no'] / 6, demo['p_pred'], 1,
                where=demo['p_pred'] < 0.5,  color=ACCENT,    alpha=0.2)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1)
ax.set_xlabel('Overs completed')
ax.set_ylabel('Live win probability for chasing team')
ax.set_title(f'Win-Probability Replay — {demo_meta["team1"]} vs {demo_meta["team2"]}')
ax.set_xlim(0, 20); ax.set_ylim(0, 1.0)
ax.legend()
savefig('live_03_replay.png')
plt.show()

Match: Mumbai Indians vs Delhi Capitals
Venue: Brabourne Stadium  (2022)
Target: 178  |  Winner: Delhi Capitals
Chase outcome: WON


## 6.7 Takeaways for the report

* The live in-match model is a *different* problem from pre-match forecasting — it operates after most of the variance has resolved.
* Accuracy is **not constant across the chase**: ~65% early, ~90% late. Quoting the headline number without the curve would be misleading.
* The calibration plot is the safer summary for stakeholder communication — it tells you whether the probabilities mean what they say.
* For broadcast deployment, this model is the one to use; for pre-match strategy or auction analytics, the pre-match models (notebook 04) are appropriate.